In [2]:
# src/ 側のコードを編集したら自動で再読み込みする
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

from racing_prediction.data.scrape_shutuba import fetch_shutuba

# 出馬表から出走馬情報を取得 (User-Agent付き / アクセスごとに1~3秒ランダムsleep)
race_id = "202605050507"
df_shutuba = fetch_shutuba(race_id)
df_shutuba

,race_id,race_date,race_venue,race_surface,race_distance,waku,umaban,horse_id,horse_name,sex,...,jockey_id,jockey_name,trainer_id,trainer_name,trainer_area,horse_weight,weight_diff,is_canceled,rank,rank_text
0,202605050507,2026-11-22,東京,dirt,2100,1,1,2022105585,クラリネットソナタ,牝,...,01091,丹内,01208,柄崎,美浦,500,10,False,13,13
1,202605050507,2026-11-22,東京,dirt,2100,1,2,2023101644,オルブライト,牡,...,01158,野中,01008,杉浦,美浦,498,-2,False,3,3
2,202605050507,2026-11-22,東京,dirt,2100,2,3,2022105400,ピティロディア,牡,...,01162,木幡巧,01221,室井,美浦,462,8,False,8,8
3,202605050507,2026-11-22,東京,dirt,2100,2,4,2023107275,パッセージピーク,セ,...,01161,菊沢,01223,三樹,美浦,504,4,False,5,5
4,202605050507,2026-11-22,東京,dirt,2100,3,5,2023105255,オメガディコン,牡,...,05203,岩田康,01204,浅利,美浦,534,4,False,14,14
5,202605050507,2026-11-22,東京,dirt,2100,3,6,2023104606,トゥントゥントゥン,牡,...,01194,大久保,01134,高橋文,美浦,432,14,False,15,15
6,202605050507,2026-11-22,東京,dirt,2100,4,7,2023106847,ヴァレンティーニ,牝,...,01092,津村,01116,新開,美浦,498,8,False,6,6
7,202605050507,2026-11-22,東京,dirt,2100,4,8,2023105726,ゴールデンダガー,牡,...,01203,石田,01221,室井,美浦,490,4,False,16,16
8,202605050507,2026-11-22,東京,dirt,2100,5,9,2023103731,モデレートゲイル,牡,...,00733,吉田豊,01054,戸田,美浦,522,6,False,1,1
9,202605050507,2026-11-22,東京,dirt,2100,5,10,2023106744,ヴァルカンテソーロ,牡,...,05626,マーカンド,01195,嘉藤,美浦,444,-2,False,11,11


In [ ]:
# 各馬の戦績ページから条件別 (距離・馬場状態・競馬場 × 芝/ダート) の平均着順を追加
# 対象レース当日以降の戦績は集計に含めない. 1頭ごとにアクセスするので時間がかかる
from racing_prediction.features.build_features import add_avg_rank_features

df_horses = add_avg_rank_features(df_shutuba)
df_horses

In [ ]:
# 各馬の血統ページから 父・母・父父・父母・母父・母母 (馬名とID) と父の系統を追加
from racing_prediction.data.scrape_horse_pedigree import add_pedigree

df_horses = add_pedigree(df_horses)
df_horses

In [3]:
# 特徴量 + 目的変数 (is_top3) + 着順 のテーブルを作成し CSV に保存
# race_ids に複数のレースを入れると縦に結合して 1 つの CSV になる
from racing_prediction.data.make_dataset import make_dataset

race_ids = ["202605050507"]
output_path = Path.cwd().parent / "data" / "processed" / "dataset.csv"

df_dataset = make_dataset(race_ids, output_path)
df_dataset

[1/1] race_id=202605050507 を取得中...
16 行を c:\Users\sakum\OneDrive\ドキュメント\GitHub\racing_prediction_reboot\data\processed\dataset.csv に保存しました


,race_id,race_date,umaban,race_venue,race_distance_category,race_surface,horse_name,sex,age,weight_carried,...,avg_rank_dirt_overseas,sire,sire_line,dam,sire_sire,sire_dam,dam_sire,dam_dam,is_top3,rank
0,202605050507,2026-11-22,1,東京,mid,dirt,クラリネットソナタ,牝,4,56.0,...,NaN,ナダル,Roberto系,レニーズゴットジップ,Blame,Ascending Angel,City Zip,No Dress Code,0,13
1,202605050507,2026-11-22,2,東京,mid,dirt,オルブライト,牡,3,56.0,...,NaN,オルフェーヴル,Halo系,エナジャイズ,ステイゴールド,オリエンタルアート,キングカメハメハ,アリー,1,3
2,202605050507,2026-11-22,3,東京,mid,dirt,ピティロディア,牡,4,58.0,...,NaN,キンシャサノキセキ,Halo系,ダノンフローラ,フジキセキ,ケルトシャーン,アグネスタキオン,ファンジカ,0,8
3,202605050507,2026-11-22,4,東京,mid,dirt,パッセージピーク,セ,3,56.0,...,NaN,アドマイヤマーズ,Halo系,メリート,ダイワメジャー,ヴィアメディチ,Redoute's Choice,Cloister,0,5
4,202605050507,2026-11-22,5,東京,mid,dirt,オメガディコン,牡,3,56.0,...,NaN,ポエティックフレア,Sadler's Wells系,ラグナグ,Dawn Approach,Maria Lee,ハービンジャー,プリンセスルシータ,0,14
5,202605050507,2026-11-22,6,東京,mid,dirt,トゥントゥントゥン,牡,3,53.0,...,NaN,ダノンバラード,Halo系,エアビオンディ,ディープインパクト,レディバラード,ロードカナロア,エアマスカット,0,15
6,202605050507,2026-11-22,7,東京,mid,dirt,ヴァレンティーニ,牝,3,54.0,...,NaN,ナダル,Roberto系,アラゴネーゼ,Blame,Ascending Angel,ゴールドアリュール,アイランドファッション,0,6
7,202605050507,2026-11-22,8,東京,mid,dirt,ゴールデンダガー,牡,3,54.0,...,NaN,フィエールマン,Halo系,スサーナトウショウ,ディープインパクト,リュヌドール,ロックオブジブラルタル,シーイズトウショウ,0,16
8,202605050507,2026-11-22,9,東京,mid,dirt,モデレートゲイル,牡,3,56.0,...,NaN,ドレフォン,Storm Bird系,レインボーダリア,Gio Ponti,Eltimaas,ブライアンズタイム,アローム,1,1
9,202605050507,2026-11-22,10,東京,mid,dirt,ヴァルカンテソーロ,牡,3,56.0,...,NaN,ゴールドシップ,Halo系,チェストケリリー,ステイゴールド,ポイントフラッグ,Scat Daddy,Santa Rosalia,0,11


In [ ]:
# 競馬場・期間・レース番号を指定して連続取得し, レースごとに CSV 保存
# 保存先: data/processed/races/{競馬場}/{開催日}/{レース番号}R.csv  (例: tokyo/20261122/07R.csv)
# 保存済みのレースは取得せずに読み込むので, 途中で止めても再実行すれば続きから取得できる
from racing_prediction.data.make_dataset import collect_races

venue = "東京"                # 札幌・函館・福島・新潟・東京・中山・中京・京都・阪神・小倉
start_date = "2026-11-01"     # 集計期間 (開始日)
end_date = "2026-11-22"       # 集計期間 (終了日)
start_race_number = 7
end_race_number = 12
race_numbers = range(start_race_number, end_race_number + 1)   # 7R〜12R

df_races = collect_races(
    venue,
    start_date,
    end_date,
    race_numbers=race_numbers,
    output_dir=Path.cwd().parent / "data" / "processed" / "races",
)